# Data preparation (CPU session — free, no GPU quota)

1. Download HyperKvasir unlabeled (~24 GB) into `/kaggle/tmp` scratch and stream-resize to 256px (~3 GB), then publish as a private Kaggle Dataset.
2. Perceptual-hash the corpus against Kvasir-SEG and exclude near-duplicates — this prevents pretraining/test contamination and the count goes in the thesis.
3. Generate the group-aware, stratified Kvasir-SEG splits.

**Set the accelerator to None.** This is pure CPU work and a GPU session would burn quota for nothing.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')
    return r

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# ~30-60 min. Publishes morsalin101/hyperkvasir-unlabeled-256.
# Requires KAGGLE_USERNAME / KAGGLE_KEY under Add-ons -> Secrets.
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')

sh('python scripts/build_hyperkvasir_dataset.py --publish morsalin101/hyperkvasir-unlabeled-256')


In [ ]:
# The labelled split (10,662 images, 23 classes) — ~400 MB, a few minutes.
# Never trained on; used only by the k-NN / linear probe in the analysis
# notebook, which is the cheapest check that pretraining actually worked.
sh('python scripts/build_hyperkvasir_dataset.py --split labeled --publish morsalin101/hyperkvasir-labeled-256')


In [ ]:
# Point at the corpus we just built in /kaggle/working — the published dataset
# only appears under /kaggle/input on a *later* session, so auto-resolution
# would not find it yet in this one.
sh('python scripts/dedup_phash.py --threshold 6 --pretrain-root /kaggle/working/hk256')


In [ ]:
sh('python -m src.data.splits')


In [ ]:
print(open('splits/dedup_report.json').read())
print(open('splits/800_100_100/stats.json').read())


### Copy `splits/` back into git — required before segmentation

The later notebooks clone the repo from GitHub, so the split files and the dedup exclusion list have to be **committed**, not just present here. The cell below copies them to `/kaggle/working/splits_to_commit/`; download that from the notebook's Output tab (or `kaggle kernels output`), drop it into `splits/` locally, and push.

This is what guarantees every run — yours and anyone reproducing it — uses byte-identical splits.

In [ ]:
import shutil
shutil.copytree('splits', '/kaggle/working/splits_to_commit', dirs_exist_ok=True)
for root, _, files in os.walk('/kaggle/working/splits_to_commit'):
    for f in sorted(files):
        print(os.path.join(root, f))
print('\nRetrieve with:\n'
      '  kaggle kernels output morsalin101/data-prep -p /tmp/dp\n'
      '  cp -r /tmp/dp/splits_to_commit/* splits/\n'
      '  git add splits && git commit -m \'data: splits + dedup list\' && git push')
